*Import Library*

In [ ]:
import gymnasium as gym
import numpy as np
import metaworld
import random
import os
os.environ['MUJOCO_GL']='osmesa'
import gymnasium as gym
from gymnasium.wrappers import ResizeObservation
from PIL import Image
import matplotlib.pyplot as plt
import importlib
import imageio
import time
from IPython.display import HTML
from base64 import b64encode


In [5]:
# Helper function to get the policy class for a given environment
def get_policy_class(env_name):
    # Map environment names to their corresponding policy class names
    env_to_policy = {
        'reach-v2': 'SawyerReachV2Policy',
        'push-v2': 'SawyerPushV2Policy',
        'pick-place-v2': 'SawyerPickPlaceV2Policy',
        'door-open-v2': 'SawyerDoorOpenV2Policy',
        'drawer-close-v2': 'SawyerDrawerCloseV2Policy',
        'drawer-open-v2': 'SawyerDrawerOpenV2Policy',
        'button-press-topdown-v2': 'SawyerButtonPressTopdownV2Policy',
        'window-open-v2': 'SawyerWindowOpenV2Policy',
        'window-close-v2': 'SawyerWindowCloseV2Policy',
        'peg-insert-side-v2': 'SawyerPegInsertionSideV2Policy'
    }
    
    if env_name not in env_to_policy:
        raise ValueError(f"No policy found for environment {env_name}")
    
    # Convert environment name to import path format
    policy_name = env_to_policy[env_name]
    # Import the policy class
    module = importlib.import_module(f"metaworld.policies.sawyer_{env_name.replace('-', '_')}_policy")
    return getattr(module, policy_name)

In [6]:
class RandomizeInitialPositionWrapper(gym.Wrapper):
    """A wrapper that randomizes the initial position and orientation of the hand and object in MetaWorld environments."""
    
    def __init__(self, env):
        super().__init__(env)
        
    def reset(self, seed=None, options=None):
        # Get the actual SawyerXYZEnv instance
        if hasattr(self.env, 'env'):
            sawyer_env = self.env.env
        else:
            sawyer_env = self.env
            
        # First, enable randomization for object position
        if hasattr(sawyer_env, '_freeze_rand_vec'):
            original_freeze = sawyer_env._freeze_rand_vec
            original_seeded = sawyer_env.seeded_rand_vec
            
            sawyer_env._freeze_rand_vec = False  # Allow randomization
            sawyer_env.seeded_rand_vec = True    # Use seeded randomization
        
        # Reset environment
        obs, info = self.env.reset(seed=seed, options=options)
        
        # Now let's randomize hand position
        if hasattr(sawyer_env, 'hand_low') and hasattr(sawyer_env, 'hand_high'):
            # Get hand position bounds
            hand_low = sawyer_env.hand_low
            hand_high = sawyer_env.hand_high
            
            # Generate random hand position within bounds
            random_hand_pos = np.random.uniform(hand_low, hand_high)
            print("Setting random hand position:", random_hand_pos)
            
            # Direct method to set hand position through mocap
            mocap_id = sawyer_env.model.body_mocapid[sawyer_env.data.body("mocap").id]
            sawyer_env.data.mocap_pos[mocap_id] = random_hand_pos
            sawyer_env.data.mocap_quat[mocap_id] = np.array([1, 0, 1, 0])
            
            # Run simulation steps to apply the changes
            for _ in range(10):
                sawyer_env.do_simulation([-1, 1], sawyer_env.frame_skip)
            
            # Update the observation to reflect new hand position
            obs = sawyer_env._get_obs()
            print("New hand position (from obs):", obs[:3])
            
        
        
        # Restore randomization settings
        if hasattr(sawyer_env, '_freeze_rand_vec'):
            sawyer_env._freeze_rand_vec = original_freeze
            sawyer_env.seeded_rand_vec = original_seeded
            
        return obs, info
    
    def _random_unit_quaternion(self):
        """Generate a random unit quaternion (uniformly distributed orientation)."""
        # Generate random values between 0 and 1
        u1, u2, u3 = np.random.random(3)
        
        # Convert to quaternion using algorithm from Ken Shoemake's paper
        q = np.zeros(4)
        q[0] = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
        q[1] = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
        q[2] = np.sqrt(u1) * np.sin(2 * np.pi * u3)
        q[3] = np.sqrt(u1) * np.cos(2 * np.pi * u3)
        
        # Normalize to ensure it's a unit quaternion
        q = q / np.linalg.norm(q)
        
        return q
        
    def _constrained_random_quaternion(self):
        """Generate a random quaternion that's more likely to keep the object upright."""
        # For push-v2, we want to primarily rotate around the Z-axis
        # This keeps the object visible and prevents it from clipping through the floor
        
        # Random rotation angle around z-axis (full 360 degrees)
        angle_z = np.random.uniform(0, 2 * np.pi)
        
        # Limited tilt angles (small rotations around x and y axes)
        # Keeping these small helps ensure the object stays upright
        angle_x = np.random.uniform(-0.2, 0.2)  # Limit x-tilt to ±0.2 radians
        angle_y = np.random.uniform(-0.2, 0.2)  # Limit y-tilt to ±0.2 radians
        
        # Convert to quaternion components
        from scipy.spatial.transform import Rotation
        
        # Create rotation from Euler angles (x, y, z order)
        rot = Rotation.from_euler('xyz', [angle_x, angle_y, angle_z])
        
        # Get quaternion and return in w, x, y, z format
        return rot.as_quat()  # Returns in x, y, z, w format

In [9]:
# Example usage:
env_name = 'reach-v2'
mt10 = metaworld.MT10(42)
env = mt10.train_classes[env_name](render_mode="rgb_array", camera_name="corner")
env_task_indices = [i for i, task in enumerate(mt10.train_tasks) if task.env_name == env_name]
# print(mt10.train_tasks)
task = mt10.train_tasks[random.choice(env_task_indices)]
env.set_task(task)

# first randomize positions
env = RandomizeInitialPositionWrapper(env)

policy = get_policy_class(env_name)()

In [ ]:
class MakeGoalObservableWrapper(gym.Wrapper):
    """A wrapper that makes the goal position observable in the environment observation."""
    
    def __init__(self, env):
        super().__init__(env)
        # Get the actual SawyerXYZEnv instance
        if hasattr(self.env, 'env'):
            sawyer_env = self.env.env
        else:
            sawyer_env = self.env
            
        # Make goal observable by setting partially_observable to False
        sawyer_env._partially_observable = False
        
        # Update observation space to reflect that goals are now visible
        # This is important if your RL algorithm checks the observation space bounds
        obs_space = sawyer_env.observation_space
        
        # Update the goal portion of observation space (last 3 elements) if needed
        if sawyer_env.goal_space is not None:
            high = obs_space.high.copy()
            low = obs_space.low.copy()
            
            # Set the goal bounds (last 3 elements)
            high[-3:] = sawyer_env.goal_space.high
            low[-3:] = sawyer_env.goal_space.low
            
            # Create updated observation space
            self.observation_space = gym.spaces.Box(
                low=low, 
                high=high,
                dtype=obs_space.dtype
            )

In [18]:
import imageio
# Set up the environment with both wrappers
env_name = 'drawer-open-v2'
env = mt10.train_classes[env_name](render_mode="rgb_array", camera_name="corner")
env_task_indices = [i for i, task in enumerate(mt10.train_tasks) if task.env_name == env_name]
task = mt10.train_tasks[random.choice(env_task_indices)]
env.set_task(task)

# Apply wrappers
env = RandomizeInitialPositionWrapper(env)
# env = MakeGoalObservableWrapper(env)

# Create policy
policy = get_policy_class(env_name)()

# Run an episode with the expert policy and save frames
def run_expert_policy_episode(env, policy, max_steps=200):
    frames = []
    obs, _ = env.reset()
    done = False
    step = 0
    total_reward = 0
    
    while not done and step < max_steps:
        # Get action from the expert policy
        action = policy.get_action(obs)
        
        # Execute action
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        
        # Render and save frame
        frame = env.render()
        # Flip the frame vertically to correct the orientation
        frame = np.flipud(frame)
        frames.append(frame)
        
        step += 1
    
    print(f"Episode finished after {step} steps. Total reward: {total_reward}")
    return frames

# Run episode and collect frames
print(f"Running expert policy for {env_name}...")
frames = run_expert_policy_episode(env, policy)

# Save video
video_path = f"expert_policy_{env_name}.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Video saved to {video_path}")

